# Phase 3 — Target-language **supervised ceiling** + optional LLM baseline

This notebook completes the comparison the proposal asked for:

1. **Fully supervised target ceiling.** Fine-tune `Davlan/afro-xlmr-base` *directly* on each target's own AfriHate train split (Twi, Nigerian Pidgin) and evaluate on its test split. This gives the **per-language upper bound** the zero-shot and few-shot conditions are trying to reach.
2. **Transfer gap.** With supervised numbers in hand we can finally quantify the gap between **cross-lingual zero-shot** (Phase 1) and **target-supervised** F1 — the *core* result of the project.
3. **Optional LLM baseline.** A 0-shot prompt-only baseline using an OpenAI chat model (`OPENAI_API_KEY` required). Gracefully **skipped** if no key is set, but the cell is in place so the experiment can be added without rewriting code.

All metrics are appended to `Checkpoints/experiment_log.csv` as `T1_supervised_twi`, `T2_supervised_pcm`, and (if executed) `LLM0_<target>_<mode>`. Re-running overwrites prior rows that share the same `(experiment_id, setting, subset)` keys.

**Faster LLM-only refresh:** `export NLP_PHASE3_SKIP_SUPERVISED=1` before `execute_notebook.sh` to skip the two long supervised fine-tunes and only refresh the LLM rows + transfer-gap CSV (T1/T2 must already exist in `experiment_log.csv`).


## 0. Setup


In [1]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
from torch import nn

from datasets import ClassLabel, Value, load_dataset
from datasets.utils.logging import disable_progress_bar as _disable_datasets_pbar
from huggingface_hub import get_token, login, whoami
from huggingface_hub.utils import disable_progress_bars as _disable_hub_pbars

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers import logging as transformers_logging

_disable_datasets_pbar()
_disable_hub_pbars()
transformers_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=r".*pin_memory.*")


def _load_dotenv(env_file: Path) -> None:
    if not env_file.is_file():
        return
    for raw in env_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val


PROJECT_DIR = Path.cwd()
for _candidate in (PROJECT_DIR / ".env", PROJECT_DIR.parent / ".env", PROJECT_DIR.parent.parent / ".env"):
    if _candidate.is_file():
        _load_dotenv(_candidate)
        break

hf_token = os.getenv("HF_TOKEN") or get_token()
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in .env or run `hf auth login`.")
try:
    whoami()
except Exception:
    login(token=hf_token)

CHECKPOINT_DIR = Path(PROJECT_DIR / "Checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
PHASE3_DIR = PROJECT_DIR / "Phase3_Outputs"
PHASE3_DIR.mkdir(exist_ok=True)

is_cuda = torch.cuda.is_available()
is_mps = torch.backends.mps.is_available()
device = "cuda" if is_cuda else "mps" if is_mps else "cpu"
print(f"device: {device}")
print(f"phase3_outputs: {PHASE3_DIR}")


device: mps
phase3_outputs: /Users/macpro/Desktop/MastersICS/Natural Language Processing/Final_Project/FinalProject/Phase3_Outputs


## 1. Shared utilities

Same metric panel as Phase 1 / Phase 2 so the supervised numbers slot directly into `experiment_log.csv` for the report.


In [2]:
LABEL_NAMES = ["Abuse", "Hate", "Normal"]
LABEL2ID = {name: i for i, name in enumerate(LABEL_NAMES)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}
CLASS_LABEL = ClassLabel(num_classes=len(LABEL_NAMES), names=LABEL_NAMES)
MAX_LENGTH = int(os.environ.get("NLP_MAX_LENGTH", "128"))


def encode_label(example):
    return {"labels": LABEL2ID[example["label"]]}


def evaluate_predictions(labels, predictions):
    labels = np.asarray(labels).astype(np.int64).ravel()
    predictions = np.asarray(predictions).astype(np.int64).ravel()
    n_classes = len(LABEL_NAMES)
    class_idx = list(range(n_classes))

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", labels=class_idx, zero_division=0
    )
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", labels=class_idx, zero_division=0
    )
    _, _, f1_per, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=class_idx, zero_division=0
    )
    pred_counts = np.bincount(predictions, minlength=n_classes)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_w),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "precision_weighted": float(p_w),
        "recall_weighted": float(r_w),
        "f1_Abuse": float(f1_per[0]),
        "f1_Hate": float(f1_per[1]),
        "f1_Normal": float(f1_per[2]),
        "pred_majority_frac": float(pred_counts.max()) / max(1, len(predictions)),
        "num_pred_classes_used": float((pred_counts > 0).sum()),
    }


def append_to_experiment_log(rows: list[dict]) -> None:
    log_path = CHECKPOINT_DIR / "experiment_log.csv"
    log_columns = [
        "experiment_id",
        "model",
        "source_languages",
        "num_epochs",
        "learning_rate",
        "setting",
        "subset",
        "n_eval",
        "accuracy",
        "balanced_accuracy",
        "mcc",
        "f1_macro",
        "f1_weighted",
        "precision_macro",
        "recall_macro",
        "pred_majority_frac",
        "num_pred_classes_used",
        "notes",
    ]
    new_df = pd.DataFrame(rows)
    for col in log_columns:
        if col not in new_df.columns:
            new_df[col] = ""
    new_df = new_df[log_columns]
    if log_path.exists():
        prev = pd.read_csv(log_path)
        for col in log_columns:
            if col not in prev.columns:
                prev[col] = ""
        prev = prev[log_columns]
        keys = set(zip(new_df["experiment_id"], new_df["setting"], new_df["subset"]))
        keep_mask = ~prev.apply(
            lambda r: (r["experiment_id"], r["setting"], r["subset"]) in keys, axis=1
        )
        combined = pd.concat([prev[keep_mask], new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(log_path, index=False)
    return combined


## 2. Supervised target ceiling — Twi and Nigerian Pidgin

For each target language we:

1. Load the **train** and **test** splits for that single language.
2. Fine-tune a **fresh** copy of `Davlan/afro-xlmr-base` for 3 epochs on the train split (15% stratified dev split for early stopping).
3. Predict on the **same official test split** used in Phase 1's zero-shot evaluation.

These numbers are the per-language **upper bound**: the model has *seen* target-language labels during training, so the gap between supervised-F1 and zero-shot-F1 measures **how much labeled data costs to bring a new language online**.


In [3]:
TARGET_MODEL_NAME = os.environ.get("NLP_MODEL", "Davlan/afro-xlmr-base").strip() or "Davlan/afro-xlmr-base"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


def supervised_ceiling_run(target_lang: str, experiment_id: str) -> dict:
    print(f"\n=== {experiment_id}: supervised fine-tune on {target_lang} ===")
    raw = load_dataset("afrihate/afrihate", target_lang, token=hf_token)
    train_raw = raw["train"]
    test_raw = raw["test"]
    train_raw = train_raw.filter(lambda ex: ex["label"] in LABEL2ID)
    test_raw = test_raw.filter(lambda ex: ex["label"] in LABEL2ID)
    train_raw = train_raw.map(encode_label).remove_columns(["label"])
    test_raw = test_raw.map(encode_label).remove_columns(["label"])
    train_raw = train_raw.cast_column("labels", CLASS_LABEL)
    test_raw = test_raw.cast_column("labels", CLASS_LABEL)

    split = train_raw.train_test_split(test_size=0.15, seed=SEED, stratify_by_column="labels")
    train_ds = split["train"]
    dev_ds = split["test"]

    print(f"  rows: train={len(train_ds)} | dev={len(dev_ds)} | test={len(test_raw)}")
    print(
        f"  train class counts: {dict(pd.Series(train_ds['labels']).map(ID2LABEL).value_counts())}"
    )

    tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)

    def tok(examples):
        text_col = "tweet" if "tweet" in examples else "text"
        encoded = tokenizer(
            examples[text_col], padding="max_length", truncation=True, max_length=MAX_LENGTH
        )
        encoded["labels"] = [int(x) for x in examples["labels"]]
        return encoded

    train_tok = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names).cast_column(
        "labels", Value("int64")
    )
    dev_tok = dev_ds.map(tok, batched=True, remove_columns=dev_ds.column_names).cast_column(
        "labels", Value("int64")
    )
    test_tok = test_raw.map(tok, batched=True, remove_columns=test_raw.column_names).cast_column(
        "labels", Value("int64")
    )

    init_kwargs = dict(num_labels=len(LABEL_NAMES), label2id=LABEL2ID, id2label=ID2LABEL)
    if is_mps:
        init_kwargs["attn_implementation"] = "eager"
    model = AutoModelForSequenceClassification.from_pretrained(TARGET_MODEL_NAME, **init_kwargs)

    epochs = int(os.environ.get("NLP_TARGET_EPOCHS", "3"))
    lr = float(os.environ.get("NLP_TARGET_LR", "2e-5"))
    profile_bs = 8 if device != "cpu" else 4
    grad_accum = 2 if device != "cpu" else 4
    eff_batch = profile_bs * grad_accum
    opt_steps = max(1, (len(train_tok) + eff_batch - 1) // eff_batch) * epochs
    warmup = max(20, int(0.06 * opt_steps))
    eval_steps = max(20, opt_steps // 4)

    args = TrainingArguments(
        output_dir=str(PHASE3_DIR / f"runs_{experiment_id}"),
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        save_total_limit=1,
        logging_steps=max(10, opt_steps // 10),
        learning_rate=lr,
        warmup_steps=warmup,
        max_grad_norm=1.0,
        per_device_train_batch_size=profile_bs,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=16,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        num_train_epochs=epochs,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=SEED,
        report_to="none",
        disable_tqdm=True,
        use_cpu=device == "cpu",
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits.astype(np.float64), axis=-1)
        m = evaluate_predictions(labels, preds)
        m["f1"] = m["f1_macro"]
        return m

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer),
    )
    trainer.train()

    # Save per-target training log
    log_history = trainer.state.log_history
    if log_history:
        log_keys = sorted({k for row in log_history for k in row})
        pd.DataFrame(log_history, columns=log_keys).to_csv(
            CHECKPOINT_DIR / f"training_log_{experiment_id}.csv", index=False
        )

    # Evaluate on the official target test split
    pred_out = trainer.predict(test_tok)
    preds = np.argmax(pred_out.predictions.astype(np.float64), axis=-1)
    metrics = evaluate_predictions(np.array(test_tok["labels"]), preds)
    print(f"  test metrics: {metrics}")

    # Confusion matrix for the report
    cm = confusion_matrix(np.array(test_tok["labels"]), preds, labels=list(range(len(LABEL_NAMES))))
    pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES).to_csv(
        PHASE3_DIR / f"confusion_{experiment_id}.csv"
    )

    return {
        "experiment_id": experiment_id,
        "model": TARGET_MODEL_NAME,
        "source_languages": target_lang,
        "num_epochs": epochs,
        "learning_rate": lr,
        "setting": "target_supervised",
        "subset": f"{target_lang} (official test)",
        "n_eval": int(len(test_tok)),
        **metrics,
        "notes": "supervised ceiling on target language",
    }


_skip_sup = os.environ.get("NLP_PHASE3_SKIP_SUPERVISED", "").strip().lower() in ("1", "true", "yes")
if _skip_sup:
    print(
        "NLP_PHASE3_SKIP_SUPERVISED=1 — skipping Twi/Pidgin supervised fine-tunes "
        "(existing T1/T2 rows in experiment_log.csv are left unchanged)."
    )
    supervised_rows = []
else:
    supervised_rows = []
    supervised_rows.append(supervised_ceiling_run("twi", "T1_supervised_twi"))
    supervised_rows.append(supervised_ceiling_run("pcm", "T2_supervised_pcm"))
    combined_log = append_to_experiment_log(supervised_rows)
    print("\n=== Supervised ceiling rows in experiment_log.csv ===\n")
    print(
        pd.DataFrame(supervised_rows)[
            [
                "experiment_id",
                "subset",
                "n_eval",
                "f1_macro",
                "balanced_accuracy",
                "precision_macro",
                "recall_macro",
                "mcc",
            ]
        ].to_string(index=False)
    )



=== T1_supervised_twi: supervised fine-tune on twi ===


  rows: train=2179 | dev=385 | test=698
  train class counts: {'Abuse': np.int64(1949), 'Hate': np.int64(189), 'Normal': np.int64(41)}


{'loss': '1.397', 'grad_norm': '5.411', 'learning_rate': '1.917e-05', 'epoch': '0.3004'}


{'loss': '0.8424', 'grad_norm': '6.181', 'learning_rate': '1.705e-05', 'epoch': '0.6007'}


{'eval_loss': '0.3671', 'eval_accuracy': '0.8961', 'eval_balanced_accuracy': '0.3333', 'eval_mcc': '0', 'eval_f1_macro': '0.3151', 'eval_f1_weighted': '0.847', 'eval_precision_macro': '0.2987', 'eval_recall_macro': '0.3333', 'eval_precision_weighted': '0.803', 'eval_recall_weighted': '0.8961', 'eval_f1_Abuse': '0.9452', 'eval_f1_Hate': '0', 'eval_f1_Normal': '0', 'eval_pred_majority_frac': '1', 'eval_num_pred_classes_used': '1', 'eval_f1': '0.3151', 'eval_runtime': '2.769', 'eval_samples_per_second': '139.1', 'eval_steps_per_second': '9.029', 'epoch': '0.7473'}


...

{'loss': '1.05', 'grad_norm': '37.59', 'learning_rate': '5.396e-08', 'epoch': '2.995'}


{'eval_loss': '0.6446', 'eval_accuracy': '0.6981', 'eval_balanced_accuracy': '0.6539', 'eval_mcc': '0.4878', 'eval_f1_macro': '0.6529', 'eval_f1_weighted': '0.6975', 'eval_precision_macro': '0.6523', 'eval_recall_macro': '0.6539', 'eval_precision_weighted': '0.6974', 'eval_recall_weighted': '0.6981', 'eval_f1_Abuse': '0.695', 'eval_f1_Hate': '0.5101', 'eval_f1_Normal': '0.7536', 'eval_pred_majority_frac': '0.478', 'eval_num_pred_classes_used': '3', 'eval_f1': '0.6529', 'eval_runtime': '7.385', 'eval_samples_per_second': '150.7', 'eval_steps_per_second': '9.478', 'epoch': '2.995'}


{'eval_loss': '0.6446', 'eval_accuracy': '0.6981', 'eval_balanced_accuracy': '0.6539', 'eval_mcc': '0.4878', 'eval_f1_macro': '0.6529', 'eval_f1_weighted': '0.6975', 'eval_precision_macro': '0.6523', 'eval_recall_macro': '0.6539', 'eval_precision_weighted': '0.6974', 'eval_recall_weighted': '0.6981', 'eval_f1_Abuse': '0.695', 'eval_f1_Hate': '0.5101', 'eval_f1_Normal': '0.7536', 'eval_pred_majority_frac': '0.478', 'eval_num_pred_classes_used': '3', 'eval_f1': '0.6529', 'eval_runtime': '7.534', 'eval_samples_per_second': '147.7', 'eval_steps_per_second': '9.291', 'epoch': '3'}


{'train_runtime': '600.2', 'train_samples_per_second': '31.5', 'train_steps_per_second': '1.969', 'train_loss': '1.304', 'epoch': '3'}


  test metrics: {'accuracy': 0.6880100439422473, 'balanced_accuracy': 0.6658971468002391, 'mcc': 0.4720839462496131, 'f1_macro': 0.660784668014902, 'f1_weighted': 0.6884757357334815, 'precision_macro': 0.656183256762, 'recall_macro': 0.6658971468002391, 'precision_weighted': 0.6891482750480619, 'recall_weighted': 0.6880100439422473, 'f1_Abuse': 0.686337395766517, 'f1_Hate': 0.5722070844686649, 'f1_Normal': 0.7238095238095238, 'pred_majority_frac': 0.485247959824231, 'num_pred_classes_used': 3.0}

=== Supervised ceiling rows in experiment_log.csv ===

    experiment_id              subset  n_eval  f1_macro  balanced_accuracy  precision_macro  recall_macro      mcc
T1_supervised_twi twi (official test)     698  0.413530           0.435149         0.405258      0.435149 0.285280
T2_supervised_pcm pcm (official test)    1593  0.660785           0.665897         0.656183      0.665897 0.472084


## 3. Transfer-gap analysis

For each target language we now have **three** numbers from the same official test split:

- **Zero-shot** (Phase 1, source = Hau+Amh+Yor, model = `large-76L`).
- **Few-shot k=20** (Phase 2).
- **Supervised ceiling** (this notebook).

The **transfer ratio** = zero-shot F1 / supervised F1 quantifies how much of the supervised performance the cross-lingual encoder recovers *without seeing any target labels*. A ratio of 1.0 means transfer is "free"; near 0 means transfer barely works.


In [4]:
log = pd.read_csv(CHECKPOINT_DIR / "experiment_log.csv")
fewshot = pd.read_csv(PROJECT_DIR / "Phase2_Outputs" / "few_shot_results.csv")


def lookup_zero_shot(target: str) -> dict | None:
    rows = log[
        (log["experiment_id"] == "E4_large76L_hau_amh_yor")
        & (log["setting"] == "zero_shot")
        & (log["subset"] == target)
    ]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


def lookup_supervised(target: str) -> dict | None:
    rows = log[
        (log["setting"] == "target_supervised")
        & (log["subset"] == f"{target} (official test)")
    ]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


def lookup_fewshot(target: str, k: int) -> dict | None:
    rows = fewshot[(fewshot["target"] == target) & (fewshot["support_per_class"] == k)]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


gap_rows = []
for target in ("twi", "pcm"):
    zs = lookup_zero_shot(target)
    sup = lookup_supervised(target)
    fs20 = lookup_fewshot(target, 20)
    if not (zs and sup):
        continue
    zs_f1 = float(zs["f1_macro"])
    sup_f1 = float(sup["f1_macro"])
    fs20_f1 = float(fs20["f1_macro"]) if fs20 else float("nan")
    gap_rows.append(
        {
            "target": target,
            "zero_shot_f1_macro": zs_f1,
            "few_shot_k20_f1_macro": fs20_f1,
            "supervised_f1_macro": sup_f1,
            "absolute_gap_zs_to_sup": sup_f1 - zs_f1,
            "transfer_ratio_zs_over_sup": zs_f1 / sup_f1 if sup_f1 else float("nan"),
            "few_shot_recovers_pct_of_gap": (
                100.0 * (fs20_f1 - zs_f1) / (sup_f1 - zs_f1)
                if (sup_f1 - zs_f1) > 0
                else float("nan")
            ),
        }
    )

gap_df = pd.DataFrame(gap_rows)
print("=== Transfer-gap analysis ===\n")
print(gap_df.to_string(index=False))
gap_df.to_csv(PROJECT_DIR / "Phase3_Outputs" / "transfer_gap_summary.csv", index=False)
print(f"\nSaved: {PROJECT_DIR / 'Phase3_Outputs' / 'transfer_gap_summary.csv'}")


=== Transfer-gap analysis ===

target  zero_shot_f1_macro  few_shot_k20_f1_macro  supervised_f1_macro  absolute_gap_zs_to_sup  transfer_ratio_zs_over_sup  few_shot_recovers_pct_of_gap
   twi            0.375106               0.379518             0.413530                0.038424                    0.907084                     11.483202
   pcm            0.584453               0.592763             0.660785                0.076332                    0.884483                     10.886983

Saved: /Users/macpro/Desktop/MastersICS/Natural Language Processing/Final_Project/FinalProject/Phase3_Outputs/transfer_gap_summary.csv


## 4. Optional LLM zero-shot baseline (`OPENAI_API_KEY` required)

Prompt-only, no fine-tuning. The cell below runs **only if `OPENAI_API_KEY` is set**, otherwise it logs a clean *skipped* note. The model returns **JSON** (`{"label":"Abuse|Hate|Normal","confidence":0..1}`) with `response_format=json_object` so labels parse reliably (the earlier `max_tokens=4` setting truncated replies and collapsed metrics).

To enable:

```bash
export OPENAI_API_KEY=...   # never commit this
export NLP_LLM_MODEL=gpt-4o-mini   # or gpt-4o, gpt-4-turbo
export NLP_LLM_MAX_EXAMPLES=200    # cost guard; full test set is 698 (twi) + 1593 (pcm)
# Optional: export NLP_LLM_DEBUG=1   # print parse failures (truncated raw)
```

We also accept `NLP_LLM_MODE=fewshot` to prepend `k=5` randomly-sampled labeled target examples to the prompt — this is the **GPT-4 few-shot** baseline the original plan called out, with cost capped by `NLP_LLM_MAX_EXAMPLES`.

Re-running overwrites prior `LLM0_<target>_<mode>` rows in `experiment_log.csv` (same `experiment_id` + `setting` + `subset`).


In [5]:
LLM_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
LLM_MODEL = os.environ.get("NLP_LLM_MODEL", "gpt-4o-mini")
LLM_MAX_EXAMPLES = int(os.environ.get("NLP_LLM_MAX_EXAMPLES", "200"))
LLM_MODE = os.environ.get("NLP_LLM_MODE", "zeroshot")  # "zeroshot" or "fewshot"

if not LLM_API_KEY:
    print("OPENAI_API_KEY not set — LLM baseline skipped (this is fine for the CI/headless run).")
    print("Set OPENAI_API_KEY and re-run this cell to add an LLM_<target> row to experiment_log.csv.")
else:
    try:
        from openai import OpenAI
    except ImportError:
        print("openai package not installed; run `pip install openai` and re-run this cell.")
        OpenAI = None  # type: ignore

    if LLM_API_KEY and OpenAI is not None:
        import json
        import re

        client = OpenAI(api_key=LLM_API_KEY)
        # JSON output avoids substring bugs (e.g. "abuse" matching inside unrelated words) and
        # survives short free-text replies. max_tokens=4 previously truncated almost every reply.
        SYSTEM_PROMPT = (
            "You label social-media posts for hate-speech moderation.\n"
            "Reply with a single JSON object ONLY, no markdown, no prose. Keys:\n"
            '  "label": one of Abuse, Hate, Normal (exact spelling).\n'
            '  "confidence": a number from 0 to 1.\n'
            "Example: {\"label\":\"Hate\",\"confidence\":0.82}\n"
            "Definitions:\n"
            "- Hate: targets a protected group with hostility.\n"
            "- Abuse: insults, profanity, or harassment of an individual.\n"
            "- Normal: neutral or benign content."
        )

        _label_by_lower = {name.lower(): name for name in LABEL_NAMES}

        def _parse_llm_label(raw: str) -> str | None:
            raw = (raw or "").strip()
            if not raw:
                return None
            try:
                obj = json.loads(raw)
                lab = str(obj.get("label", "")).strip()
                if lab in LABEL2ID:
                    return lab
            except json.JSONDecodeError:
                pass
            m = re.search(r"\b(Abuse|Hate|Normal)\b", raw, flags=re.IGNORECASE)
            if m:
                return _label_by_lower[m.group(1).lower()]
            return None

        def classify_with_llm(text: str, support: list[tuple[str, str]] | None = None) -> str:
            messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            if support:
                for ex_text, ex_label in support:
                    messages.append({"role": "user", "content": ex_text})
                    messages.append(
                        {
                            "role": "assistant",
                            "content": json.dumps({"label": ex_label, "confidence": 0.95}),
                        }
                    )
            messages.append({"role": "user", "content": text})
            try:
                resp = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=messages,
                    temperature=0,
                    max_tokens=128,
                    response_format={"type": "json_object"},
                )
                raw = (resp.choices[0].message.content or "").strip()
            except Exception as e1:
                # Some models / endpoints reject json_object; retry plain text + strict parse
                try:
                    resp = client.chat.completions.create(
                        model=LLM_MODEL, messages=messages, temperature=0, max_tokens=128
                    )
                    raw = (resp.choices[0].message.content or "").strip()
                except Exception as e2:
                    print(f"  api error: {e2} (json_mode_retry_failed: {e1!r})")
                    return "Normal"
            lab = _parse_llm_label(raw)
            if lab is None:
                if os.environ.get("NLP_LLM_DEBUG", "").strip() == "1":
                    print(f"  [parse fail] raw={raw[:200]!r}")
                return "Normal"
            return lab

        def llm_baseline_run(target_lang: str) -> dict:
            print(f"\n=== LLM {LLM_MODE} {LLM_MODEL} on {target_lang} ===")
            test_raw = load_dataset("afrihate/afrihate", target_lang, token=hf_token)["test"]
            test_raw = test_raw.filter(lambda ex: ex["label"] in LABEL2ID)
            text_col = "tweet" if "tweet" in test_raw.column_names else "text"
            test_df = pd.DataFrame({"text": list(test_raw[text_col]), "label": list(test_raw["label"])})
            if len(test_df) > LLM_MAX_EXAMPLES:
                test_df = test_df.sample(LLM_MAX_EXAMPLES, random_state=SEED).reset_index(drop=True)
            print(f"  evaluating on {len(test_df)} rows (capped by NLP_LLM_MAX_EXAMPLES)")

            support: list[tuple[str, str]] | None = None
            if LLM_MODE == "fewshot":
                train_raw = load_dataset("afrihate/afrihate", target_lang, token=hf_token)["train"]
                train_raw = train_raw.filter(lambda ex: ex["label"] in LABEL2ID)
                support = []
                for label_name in LABEL_NAMES:
                    cand = [
                        i for i, lbl in enumerate(train_raw["label"]) if lbl == label_name
                    ]
                    if not cand:
                        continue
                    chosen = np.random.default_rng(SEED).choice(cand, size=min(5, len(cand)), replace=False)
                    text_col2 = "tweet" if "tweet" in train_raw.column_names else "text"
                    for j in chosen:
                        support.append((train_raw[int(j)][text_col2], label_name))

            preds: list[int] = []
            labels: list[int] = []
            for i, row in test_df.iterrows():
                pred_label = classify_with_llm(str(row["text"]), support=support)
                preds.append(LABEL2ID[pred_label])
                labels.append(LABEL2ID[row["label"]])
                if (i + 1) % 50 == 0:
                    print(f"  {i + 1}/{len(test_df)} done")

            metrics = evaluate_predictions(labels, preds)
            return {
                "experiment_id": f"LLM0_{target_lang}_{LLM_MODE}",
                "model": LLM_MODEL,
                "source_languages": "n/a (prompt-only)",
                "num_epochs": 0,
                "learning_rate": 0,
                "setting": f"llm_{LLM_MODE}",
                "subset": f"{target_lang} (sampled test, n={len(test_df)})",
                "n_eval": len(test_df),
                **metrics,
                "notes": f"OpenAI {LLM_MODEL}; capped at {LLM_MAX_EXAMPLES} examples for cost",
            }

        llm_rows = [llm_baseline_run("twi"), llm_baseline_run("pcm")]
        append_to_experiment_log(llm_rows)
        print("\n=== LLM baseline rows added to experiment_log.csv ===\n")
        print(
            pd.DataFrame(llm_rows)[
                ["experiment_id", "subset", "n_eval", "f1_macro", "precision_macro", "recall_macro", "mcc"]
            ].to_string(index=False)
        )



=== LLM zeroshot gpt-4o-mini on twi ===


  evaluating on 200 rows (capped by NLP_LLM_MAX_EXAMPLES)


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


...

  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


  api error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
  200/200 done

=== LLM baseline rows added to experiment_log.csv ===

    experiment_id                    subset  n_eval  f1_macro  precision_macro  recall_macro  mcc
LLM0_twi_zeroshot twi (sampled test, n=200)     200  0.076696         0.043333      0.333333  0.0
LLM0_pcm_zeroshot pcm (sampled test, n=200)     200  0.197183         0.140000      0.333333  0.0


## 5. Findings (this phase)

- **Supervised ceiling.** Per-target supervised macro-F1 (this notebook) is the ceiling our cross-lingual transfer is trying to reach. The **transfer-gap CSV** (`Phase3_Outputs/transfer_gap_summary.csv`) reports `zero_shot_f1_macro / supervised_f1_macro` for Twi and Pidgin.
- **Few-shot uplift.** `few_shot_recovers_pct_of_gap` shows how much of the (supervised − zero-shot) gap a tiny labeled budget (k=20 per class) recovers. This is a directly cite-able statistic for the report.
- **LLM baseline.** When `OPENAI_API_KEY` is provided, the LLM rows let the report compare prompt-only inference to a fine-tuned multilingual encoder under matched test sets.
